# Parameter Optimization & Overfitting — Notebook บทที่ 5

Notebook นี้ใช้ราคา SPY รายวันจาก Webull OpenAPI ชุดเดียวกับบทที่ 4 (ไฟล์ `data/spy-webull-daily.csv`)
ไม่มีการเรียก API ใหม่ รันซ้ำกี่ครั้งก็ได้ตัวเลขเดิม ใช้เพียง Python standard library

กติกาเหมือนบทที่ 4: ถือ SPY 100% เมื่อ EMA เร็วอยู่เหนือ EMA ช้าที่ราคาปิดวันก่อน มิฉะนั้นถือเงินสด
ซื้อขายที่ราคาเปิดวันถัดไป ค่าธรรมเนียมสมมติ 0.1% ต่อข้าง ไม่มี leverage และไม่ short
Sharpe = ค่าเฉลี่ย / ส่วนเบี่ยงเบนมาตรฐานของผลตอบแทนรายวัน × √252 โดยใช้อัตราปลอดความเสี่ยง 0%

In [1]:
"""Chapter 5: Parameter Optimization & Overfitting with an EMA crossover.

Pure Python 3.9+ (standard library only). Reads the frozen SPY daily bars that
chapter 4 (Backtest Performance Evaluation) fetched with Webull OpenAPI; this
script never calls an API.

Rules shared with chapter 4:
- EMA(fast) and EMA(slow) of daily closes, seeded at the first warmup close.
- Target 100% long when EMA(fast) > EMA(slow) at the previous close, else cash.
- Trade at the next open. Fee 0.1% per side on notional. No leverage or shorting.
- Close any remaining position at the final close of the window (with fee).
- Sharpe: mean / sample SD of daily returns x sqrt(252), risk-free rate 0.

Run:  python3 lesson_15.py --output-dir output
"""

import argparse
import csv
import json
import math
import random
from pathlib import Path

# Works as a script, inside the notebook (no __file__), and in the repository.
HERE = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
DATA_CANDIDATES = [
    HERE / 'data' / 'spy-webull-daily.csv',
    HERE / 'performance' / 'data' / 'spy-webull-daily.csv',
    HERE.parent / 'python' / 'data' / 'spy-webull-daily.csv',
    HERE.parents[1] / 'lessons' / 'performance' / 'data' / 'spy-webull-daily.csv',
]

WARMUP_BARS = 400
FEE = 0.001
PERIODS_PER_YEAR = 252
EVALUATION = ('2006-09-26', '2026-09-25')
IN_SAMPLE = ('2006-09-26', '2016-09-25')
OUT_OF_SAMPLE = ('2016-09-26', '2026-09-25')
FAST_SPANS = list(range(5, 101, 5))       # 5, 10, ..., 100
SLOW_SPANS = list(range(50, 301, 10))     # 50, 60, ..., 300
BASELINE = (20, 200)
WALK_TRAIN_YEARS = 5
NOISE_SEED = 20260929
NOISE_WORLDS = 60
NOISE_DAYS = 2520                         # 10 years of 252 days: 5 in-sample + 5 out-of-sample
NOISE_FAST = [5, 10, 20, 30, 40, 50, 60, 80, 100]
NOISE_SLOW = [50, 75, 100, 125, 150, 175, 200, 250, 300]
TRIAL_COUNTS = [1, 2, 5, 10, 20, 40, 74]   # 74 = every pair in the noise grid


# ---------------------------------------------------------------- data

def load_bars(path=None):
    """Read the frozen Webull CSV: date, open, high, low, close, volume."""
    path = Path(path) if path else next(p for p in DATA_CANDIDATES if p.is_file())
    with path.open(newline='', encoding='utf-8') as f:
        rows = list(csv.DictReader(f))
    first = next(i for i, row in enumerate(rows) if row['date'] >= EVALUATION[0])
    rows = rows[first - WARMUP_BARS:]
    return [{'date': r['date'], 'open': float(r['open']), 'close': float(r['close'])} for r in rows]


def ema(values, span):
    """Exponential moving average seeded at the first value (same as chapter 4)."""
    alpha = 2 / (span + 1)
    out, level = [], None
    for value in values:
        level = value if level is None else alpha * value + (1 - alpha) * level
        out.append(level)
    return out


class EmaCache:
    """Compute each EMA span once and reuse it across the parameter grid."""

    def __init__(self, closes):
        self.closes = closes
        self.store = {}

    def __call__(self, span):
        if span not in self.store:
            self.store[span] = ema(self.closes, span)
        return self.store[span]


def window_indices(bars, start, end):
    """First and last bar index whose date lies inside [start, end]."""
    first = next(i for i, b in enumerate(bars) if b['date'] >= start)
    last = max(i for i, b in enumerate(bars) if b['date'] <= end)
    return first, last


# ---------------------------------------------------------------- backtest

def run_window(bars, fast_line, slow_line, first, last, fee=FEE, capital=100000.0, pair_at=None):
    """Long/cash account from the open of bar `first` to the close of bar `last`.

    `pair_at(i)` may return other EMA lines for bar i (walk-forward); otherwise the
    fixed lines are used. Returns daily closing equity and the number of round trips.
    """
    cash, units, trades = capital, 0.0, 0
    equity = []
    for i in range(first, last + 1):
        f_line, s_line = pair_at(i) if pair_at else (fast_line, slow_line)
        target = 1 if f_line[i - 1] > s_line[i - 1] else 0
        price = bars[i]['open']
        if target == 1 and units == 0:
            units = cash / (price * (1 + fee))
            cash = 0.0
        elif target == 0 and units > 0:
            cash = units * price * (1 - fee)
            units, trades = 0.0, trades + 1
        if i == last and units > 0:
            cash = units * bars[i]['close'] * (1 - fee)
            units, trades = 0.0, trades + 1
        equity.append(cash + units * bars[i]['close'])
    return equity, trades


def buy_and_hold(bars, first, last, fee=FEE, capital=100000.0):
    units = capital / (bars[first]['open'] * (1 + fee))
    equity = [units * bars[i]['close'] for i in range(first, last + 1)]
    equity[-1] *= 1 - fee
    return equity


def daily_returns(equity, capital=100000.0):
    previous, out = capital, []
    for value in equity:
        out.append(value / previous - 1)
        previous = value
    return out


def sharpe(returns):
    n = len(returns)
    if n < 2:
        return float('nan')
    mean = sum(returns) / n
    var = sum((r - mean) ** 2 for r in returns) / (n - 1)
    return mean / math.sqrt(var) * math.sqrt(PERIODS_PER_YEAR) if var > 0 else 0.0


def max_drawdown(equity, capital=100000.0):
    peak, worst = capital, 0.0
    for value in equity:
        peak = max(peak, value)
        worst = min(worst, value / peak - 1)
    return worst


def years_between(start, end):
    from datetime import date
    a, b = date.fromisoformat(start), date.fromisoformat(end)
    return (b - a).days / 365.25


def metrics(equity, dates, trades=None, capital=100000.0):
    rets = daily_returns(equity, capital)
    years = years_between(dates[0], dates[-1])
    total = equity[-1] / capital - 1
    out = {'totalReturn': total, 'cagr': (1 + total) ** (1 / years) - 1,
           'sharpe': sharpe(rets), 'maxDrawdown': max_drawdown(equity, capital)}
    if trades is not None:
        out['trades'] = trades
    return out


def evaluate_pair(bars, lines, fast, slow, window):
    first, last = window_indices(bars, *window)
    equity, trades = run_window(bars, lines(fast), lines(slow), first, last)
    return metrics(equity, [bars[first]['date'], bars[last]['date']], trades), equity


def grid_pairs(fast_spans=FAST_SPANS, slow_spans=SLOW_SPANS):
    return [(f, s) for f in fast_spans for s in slow_spans if f < s]


def run_grid(bars, window, pairs=None):
    lines = EmaCache([b['close'] for b in bars])
    rows = []
    for fast, slow in pairs or grid_pairs():
        m, _ = evaluate_pair(bars, lines, fast, slow, window)
        rows.append({'fast': fast, 'slow': slow, **m})
    return rows


def best(rows, key='sharpe'):
    return max(rows, key=lambda r: r[key])


def neighbourhood_mean(rows, key='sharpe'):
    """Average of each cell and its grid neighbours (one step in each direction)."""
    table = {(r['fast'], r['slow']): r[key] for r in rows}
    out = []
    for r in rows:
        fi, si = FAST_SPANS.index(r['fast']), SLOW_SPANS.index(r['slow'])
        values = [table[(FAST_SPANS[a], SLOW_SPANS[b])]
                  for a in range(fi - 1, fi + 2) for b in range(si - 1, si + 2)
                  if 0 <= a < len(FAST_SPANS) and 0 <= b < len(SLOW_SPANS)
                  and (FAST_SPANS[a], SLOW_SPANS[b]) in table]
        out.append({**r, key + 'Neighbourhood': sum(values) / len(values)})
    return out


def rank_correlation(xs, ys):
    """Spearman rank correlation without ties handling beyond average ranks."""
    def ranks(values):
        order = sorted(range(len(values)), key=lambda i: values[i])
        result = [0.0] * len(values)
        i = 0
        while i < len(order):
            j = i
            while j + 1 < len(order) and values[order[j + 1]] == values[order[i]]:
                j += 1
            for k in range(i, j + 1):
                result[order[k]] = (i + j) / 2 + 1
            i = j + 1
        return result
    rx, ry = ranks(xs), ranks(ys)
    mx, my = sum(rx) / len(rx), sum(ry) / len(ry)
    cov = sum((a - mx) * (b - my) for a, b in zip(rx, ry))
    return cov / math.sqrt(sum((a - mx) ** 2 for a in rx) * sum((b - my) ** 2 for b in ry))


# ---------------------------------------------------------------- walk-forward

def walk_forward(bars, train_years=WALK_TRAIN_YEARS, pairs=None):
    """Each year pick the best-Sharpe pair on the previous `train_years`, then trade one year.

    Training Sharpe uses each pair's continuous daily returns inside the training window.
    Trading is one account whose active pair changes only at each yearly boundary.
    """
    pairs = pairs or grid_pairs()
    lines = EmaCache([b['close'] for b in bars])
    e_first, e_last = window_indices(bars, *EVALUATION)
    continuous = {}
    for fast, slow in pairs:
        equity, _ = run_window(bars, lines(fast), lines(slow), e_first, e_last)
        continuous[(fast, slow)] = daily_returns(equity)
    dates = [bars[i]['date'] for i in range(e_first, e_last + 1)]
    start_year = int(EVALUATION[0][:4])
    steps = []
    year = start_year + train_years
    while True:
        test_start = f'{year}{EVALUATION[0][4:]}'
        test_end = f'{year + 1}-09-25'
        if test_start > EVALUATION[1]:
            break
        train_start = f'{year - train_years}{EVALUATION[0][4:]}'
        a = next(i for i, d in enumerate(dates) if d >= train_start)
        b = max(i for i, d in enumerate(dates) if d < test_start)
        scores = {pair: sharpe(continuous[pair][a:b + 1]) for pair in pairs}
        chosen = max(pairs, key=lambda p: scores[p])
        steps.append({'trainStart': dates[a], 'trainEnd': dates[b], 'testStart': test_start,
                      'testEnd': min(test_end, EVALUATION[1]), 'fast': chosen[0], 'slow': chosen[1],
                      'trainSharpe': scores[chosen]})
        year += 1
    first, last = window_indices(bars, steps[0]['testStart'], EVALUATION[1])

    def pair_at(i):
        active = steps[0]
        for step in steps:
            if bars[i]['date'] >= step['testStart']:
                active = step
        return lines(active['fast']), lines(active['slow'])

    equity, trades = run_window(bars, None, None, first, last, pair_at=pair_at)
    for step in steps:
        a, b = window_indices(bars, step['testStart'], step['testEnd'])
        segment = equity[a - first:b - first + 1]
        base = equity[a - first - 1] if a > first else 100000.0
        step['testReturn'] = segment[-1] / base - 1
    window = (bars[first]['date'], bars[last]['date'])
    return {'steps': steps, 'equity': equity, 'trades': trades, 'window': window,
            'metrics': metrics(equity, list(window), trades)}


# ---------------------------------------------------------------- noise worlds

def noise_world(rng, source_returns, days=NOISE_DAYS):
    """Resample real daily returns with replacement: same size of moves, no time order.

    Shuffling removes any trend the EMA rule could exploit, so any 'edge' found here
    comes from fitting noise. Open equals the previous close in this synthetic world.
    """
    price, closes = 100.0, []
    for _ in range(days):
        price *= 1 + rng.choice(source_returns)
        closes.append(price)
    return closes


def noise_pair_returns(closes, fast, slow, cache, fee=FEE):
    f_line, s_line = cache(fast), cache(slow)
    out, position = [], 0
    for t in range(1, len(closes)):
        r = closes[t] / closes[t - 1] - 1
        target = 1 if f_line[t - 1] > s_line[t - 1] else 0
        cost = fee if target != position else 0.0
        out.append(target * r - cost)
        position = target
    return out


def noise_experiment(bars, worlds=NOISE_WORLDS, seed=NOISE_SEED):
    """For each synthetic world: choose the best in-sample pair among N tried, read its out-of-sample Sharpe."""
    rng = random.Random(seed)
    e_first, e_last = window_indices(bars, *EVALUATION)
    closes = [b['close'] for b in bars[e_first - 1:e_last + 1]]
    source = [closes[i] / closes[i - 1] - 1 for i in range(1, len(closes))]
    pairs = [(f, s) for f in NOISE_FAST for s in NOISE_SLOW if f < s]
    half = (NOISE_DAYS - 1) // 2
    results = []
    for world in range(worlds):
        path = noise_world(rng, source)
        cache = EmaCache(path)
        order = pairs[:]
        rng.shuffle(order)
        stats = {}
        for pair in pairs:
            rets = noise_pair_returns(path, pair[0], pair[1], cache)
            stats[pair] = (sharpe(rets[:half]), sharpe(rets[half:]))
        hold = [path[t] / path[t - 1] - 1 for t in range(1, len(path))]
        picks = []
        for n in TRIAL_COUNTS:
            chosen = max(order[:n], key=lambda p: stats[p][0])
            picks.append({'trials': n, 'fast': chosen[0], 'slow': chosen[1],
                          'inSample': stats[chosen][0], 'outOfSample': stats[chosen][1]})
        results.append({'world': world + 1, 'buyHoldInSample': sharpe(hold[:half]),
                        'buyHoldOutOfSample': sharpe(hold[half:]), 'picks': picks})
    summary = []
    for k, n in enumerate(TRIAL_COUNTS):
        ins = [w['picks'][k]['inSample'] for w in results]
        outs = [w['picks'][k]['outOfSample'] for w in results]
        summary.append({'trials': n, 'meanInSample': sum(ins) / len(ins),
                        'meanOutOfSample': sum(outs) / len(outs)})
    return {'seed': seed, 'worlds': worlds, 'days': NOISE_DAYS, 'pairs': len(pairs),
            'trialCounts': TRIAL_COUNTS, 'results': results, 'summary': summary}


# ---------------------------------------------------------------- report

def build_results(bars):
    pairs = grid_pairs()
    full = run_grid(bars, EVALUATION, pairs)
    ins = neighbourhood_mean(run_grid(bars, IN_SAMPLE, pairs))
    outs = run_grid(bars, OUT_OF_SAMPLE, pairs)
    best_is = best(ins)
    plateau = max(ins, key=lambda r: r['sharpeNeighbourhood'])
    lookup_out = {(r['fast'], r['slow']): r for r in outs}
    lookup_full = {(r['fast'], r['slow']): r for r in full}
    return {
        'pairs': len(pairs), 'fastSpans': FAST_SPANS, 'slowSpans': SLOW_SPANS,
        'windows': {'evaluation': EVALUATION, 'inSample': IN_SAMPLE, 'outOfSample': OUT_OF_SAMPLE},
        'full': full, 'inSample': ins, 'outOfSample': outs,
        'bestFull': best(full), 'bestInSample': best_is, 'plateauInSample': plateau,
        'bestInSampleOut': lookup_out[(best_is['fast'], best_is['slow'])],
        'plateauOut': lookup_out[(plateau['fast'], plateau['slow'])],
        'baselineFull': lookup_full[BASELINE], 'baselineOut': lookup_out[BASELINE],
        'rankCorrelation': rank_correlation([r['sharpe'] for r in ins],
                                            [lookup_out[(r['fast'], r['slow'])]['sharpe'] for r in ins]),
    }


def print_summary(results):
    def line(label, m):
        print(f"{label:<34} Sharpe {m['sharpe']:.2f}  CAGR {m['cagr']:.2%}  MDD {m['maxDrawdown']:.2%}  trades {m.get('trades', '-')}")
    print(f"Grid: {results['pairs']} EMA pairs, fast {FAST_SPANS[0]}-{FAST_SPANS[-1]}, slow {SLOW_SPANS[0]}-{SLOW_SPANS[-1]}")
    b = results['bestInSample']
    line(f"Best in-sample {b['fast']}/{b['slow']} (IS)", b)
    line(f"Same pair {b['fast']}/{b['slow']} (OOS)", results['bestInSampleOut'])
    line(f"Baseline 20/200 (OOS)", results['baselineOut'])
    print(f"Rank correlation IS vs OOS Sharpe: {results['rankCorrelation']:.2f}")


def main():
    parser = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    parser.add_argument('--output-dir', default='output')
    parser.add_argument('--data', default=None)
    args = parser.parse_args()
    bars = load_bars(args.data)
    results = build_results(bars)
    print_summary(results)
    wf = walk_forward(bars)
    print(f"Walk-forward {wf['window'][0]} to {wf['window'][1]}: Sharpe {wf['metrics']['sharpe']:.2f}, CAGR {wf['metrics']['cagr']:.2%}")
    noise = noise_experiment(bars)
    for row in noise['summary']:
        print(f"Noise, best of {row['trials']:>2} pairs: IS Sharpe {row['meanInSample']:.2f}, OOS Sharpe {row['meanOutOfSample']:.2f}")
    out = Path(args.output_dir)
    out.mkdir(parents=True, exist_ok=True)
    (out / 'optimization-results.json').write_text(json.dumps(
        {'grid': results, 'walkForward': {k: v for k, v in wf.items() if k != 'equity'}, 'noise': noise},
        ensure_ascii=False, indent=1), encoding='utf-8')
    print(f"Saved {out / 'optimization-results.json'}")



print('Loaded lesson 15 definitions; grid', len(grid_pairs()), 'pairs')

Loaded lesson 15 definitions; grid 484 pairs


## 1. โหลดข้อมูลและตรวจว่าตรงกับบทที่ 4

EMA 20/200 ช่วง 2006-09-26 ถึง 2026-09-25 ต้องได้ผลตอบแทนสะสม 367.41% และ 17 เทรดเท่ากับบทที่ 4

In [2]:
bars = load_bars()
lines = EmaCache([b['close'] for b in bars])
baseline, _ = evaluate_pair(bars, lines, 20, 200, EVALUATION)
print('Bars loaded:', len(bars), 'first', bars[0]['date'], 'last', bars[-1]['date'])
print({k: round(v, 4) for k, v in baseline.items()})
assert abs(baseline['totalReturn'] - 3.674102) < 1e-6 and baseline['trades'] == 17

Bars loaded: 5431 first 2005-02-24 last 2026-09-25
{'totalReturn': 3.6741, 'cagr': 0.0802, 'sharpe': 0.6567, 'maxDrawdown': -0.2667, 'trades': 17}


## 2. Grid search ทั้ง 20 ปี

ลอง EMA เร็ว 5–100 (ทีละ 5) คู่กับ EMA ช้า 50–300 (ทีละ 10) โดยให้เส้นเร็วสั้นกว่าเส้นช้า รวม 484 คู่
แล้วเรียงตาม Sharpe ของทั้ง 20 ปี ตัวเลขนี้คือผลที่ได้จากการมองย้อนหลังทั้งหมด

In [3]:
full = run_grid(bars, EVALUATION)
top = sorted(full, key=lambda r: -r['sharpe'])[:5]
for r in top:
    print(f"EMA {r['fast']:>3}/{r['slow']:<3} Sharpe {r['sharpe']:.3f} CAGR {r['cagr']:.2%} MDD {r['maxDrawdown']:.2%} trades {r['trades']}")
print('Pairs tested:', len(full))

EMA   5/90  Sharpe 0.841 CAGR 9.49% MDD -19.01% trades 46
EMA  10/60  Sharpe 0.834 CAGR 9.33% MDD -19.10% trades 44
EMA  25/50  Sharpe 0.815 CAGR 9.32% MDD -20.12% trades 32
EMA   5/100 Sharpe 0.809 CAGR 9.13% MDD -21.94% trades 46
EMA  10/110 Sharpe 0.809 CAGR 9.28% MDD -22.46% trades 32
Pairs tested: 484


## 3. แบ่ง In-sample และ Out-of-sample

เลือกคู่ที่ดีที่สุดจากช่วงฝึก 2006-09-26 ถึง 2016-09-25 เท่านั้น แล้วนำคู่นั้นไปอ่านผลช่วงทดสอบ 2016-09-26 ถึง 2026-09-25
แต่ละช่วงเริ่มบัญชีใหม่ด้วยเงินสด 100,000 USD แต่ EMA คำนวณต่อเนื่องจากข้อมูลก่อนหน้า

In [4]:
results = build_results(bars)
print_summary(results)
out_sorted = sorted(results['outOfSample'], key=lambda r: -r['sharpe'])
rank = {(r['fast'], r['slow']): i + 1 for i, r in enumerate(out_sorted)}
b = results['bestInSample']
print('Out-of-sample rank of the in-sample winner:', rank[(b['fast'], b['slow'])], 'of', len(out_sorted))

Grid: 484 EMA pairs, fast 5-100, slow 50-300
Best in-sample 65/120 (IS)         Sharpe 0.72  CAGR 8.59%  MDD -17.31%  trades 6
Same pair 65/120 (OOS)             Sharpe 0.77  CAGR 10.42%  MDD -29.36%  trades 6
Baseline 20/200 (OOS)              Sharpe 0.83  CAGR 10.93%  MDD -26.67%  trades 7
Rank correlation IS vs OOS Sharpe: -0.09
Out-of-sample rank of the in-sample winner: 197 of 484


## 4. เทียบกับการซื้อแล้วถือในแต่ละช่วง

In [5]:
for name, window in [('In-sample', IN_SAMPLE), ('Out-of-sample', OUT_OF_SAMPLE)]:
    first, last = window_indices(bars, *window)
    hold = metrics(buy_and_hold(bars, first, last), [bars[first]['date'], bars[last]['date']])
    pick, _ = evaluate_pair(bars, lines, b['fast'], b['slow'], window)
    print(f"{name:<14} buy & hold Sharpe {hold['sharpe']:.2f} CAGR {hold['cagr']:.2%} | EMA {b['fast']}/{b['slow']} Sharpe {pick['sharpe']:.2f} CAGR {pick['cagr']:.2%}")

In-sample      buy & hold Sharpe 0.44 CAGR 7.20% | EMA 65/120 Sharpe 0.72 CAGR 8.59%
Out-of-sample  buy & hold Sharpe 0.89 CAGR 15.42% | EMA 65/120 Sharpe 0.77 CAGR 10.42%


## 5. ตลาดที่ไม่มีสัญญาณ

สุ่มผลตอบแทนรายวันของ SPY แบบใส่คืน (bootstrap) สร้างโลกสมมติ 60 โลก โลกละ 2,520 วัน ลำดับเวลาถูกทำลาย
จึงไม่มีแนวโน้มให้ EMA จับได้จริง ในแต่ละโลก ลองสุ่ม N คู่จาก 74 คู่ เลือกคู่ที่ Sharpe ครึ่งแรกดีที่สุด แล้วอ่าน Sharpe ครึ่งหลัง

In [6]:
noise = noise_experiment(bars)
for row in noise['summary']:
    print(f"best of {row['trials']:>2} pairs: in-sample Sharpe {row['meanInSample']:.2f}, out-of-sample Sharpe {row['meanOutOfSample']:.2f}")

best of  1 pairs: in-sample Sharpe 0.30, out-of-sample Sharpe 0.31
best of  2 pairs: in-sample Sharpe 0.39, out-of-sample Sharpe 0.33
best of  5 pairs: in-sample Sharpe 0.47, out-of-sample Sharpe 0.34
best of 10 pairs: in-sample Sharpe 0.50, out-of-sample Sharpe 0.32
best of 20 pairs: in-sample Sharpe 0.52, out-of-sample Sharpe 0.34
best of 40 pairs: in-sample Sharpe 0.55, out-of-sample Sharpe 0.36
best of 74 pairs: in-sample Sharpe 0.57, out-of-sample Sharpe 0.36


## 6. Walk-forward

ทุกปี เลือกคู่ที่ Sharpe ดีที่สุดใน 5 ปีก่อนหน้า แล้วใช้คู่นั้นเทรดหนึ่งปี บัญชีเดียวต่อเนื่องตั้งแต่ 2011-09-26

In [7]:
wf = walk_forward(bars)
for step in wf['steps']:
    print(f"{step['testStart']}: EMA {step['fast']:>3}/{step['slow']:<3} train Sharpe {step['trainSharpe']:.2f} -> next-year return {step['testReturn']:.2%}")
m = wf['metrics']
print(f"Walk-forward Sharpe {m['sharpe']:.2f}, CAGR {m['cagr']:.2%}, MDD {m['maxDrawdown']:.2%}, trades {m['trades']}")

2011-09-26: EMA  10/50  train Sharpe 0.76 -> next-year return 8.19%
2012-09-26: EMA  10/50  train Sharpe 0.76 -> next-year return 13.75%
2013-09-26: EMA  10/50  train Sharpe 1.08 -> next-year return 11.68%
2014-09-26: EMA  55/240 train Sharpe 1.05 -> next-year return 0.76%
2015-09-26: EMA  55/80  train Sharpe 0.98 -> next-year return -0.26%
2016-09-26: EMA 100/270 train Sharpe 1.05 -> next-year return 17.58%
2017-09-26: EMA   5/270 train Sharpe 1.13 -> next-year return 18.93%
2018-09-26: EMA   5/270 train Sharpe 1.10 -> next-year return -5.68%
2019-09-26: EMA   5/300 train Sharpe 0.78 -> next-year return 4.97%
2020-09-26: EMA   5/160 train Sharpe 0.93 -> next-year return 36.94%
2021-09-26: EMA  10/110 train Sharpe 1.31 -> next-year return -10.28%
2022-09-26: EMA  10/110 train Sharpe 0.90 -> next-year return 3.83%
2023-09-26: EMA  10/110 train Sharpe 0.77 -> next-year return 28.22%
2024-09-26: EMA  10/110 train Sharpe 1.07 -> next-year return 12.26%
2025-09-26: EMA  40/110 train Sharpe 

## อ่านผลอย่างไร

ตัวเลขทั้งหมดเป็นผลจำลองบนราคาที่ปรับย้อนหลังแล้ว ไม่รวมภาษี spread slippage และข้อจำกัดการส่งคำสั่งจริง
ผลของช่วงทดสอบชุดเดียวเป็นเพียงหนึ่งเส้นทางของประวัติศาสตร์ ไม่ได้พิสูจน์ว่าคู่ใดจะดีในอนาคต

แหล่งอ้างอิง: Yves Hilpisch, *Python for Algorithmic Trading* (O'Reilly, 2020) หน้า PDF 87–88 (optimize_parameters แบบ brute force),
100–101 (Data Snooping and Overfitting) และ 118 (in-sample fitting กับ out-of-sample evaluation)